# Learning Module: Propositional Logic RKA 25

## 1. Introduction

**Propositional logic** is the simplest formal language for representing knowledge. A *knowledge-based agent* stores facts about the world in a **Knowledge Base (KB)** and derives new conclusions through **inference**.

In this module we implement everything *from scratch*:

| Part | Content |
|---|---|
| Syntax | Proposition symbols and logical operators |
| Semantics | Models, truth tables, validity, satisfiability |
| Inference 1 | **TT-Entails** (model checking / truth-table enumeration) |
| Inference 2 | **CNF + Resolution** (proof by contradiction) |
| Inference 3 | **Forward Chaining** for Horn clauses |
| Case study | **Wumpus World** (AIMA Chapter 7) |

Reference: Russell & Norvig, *Artificial Intelligence: A Modern Approach*, Chapter 7.

## 2. Syntax of Propositional Logic

### 2.1 Symbols and Operators

An atomic sentence is a single **proposition symbol** (e.g. `P`, `Q`, `B11`) whose value is *True* or *False*. Complex sentences are built with operators:

| Operator | Symbol | Name | In Python (this module) | Read as |
|---|---|---|---|---|
| Negation | ¬ | NOT | `~P` | "not P" |
| Conjunction | ∧ | AND | `P & Q` | "P and Q" |
| Disjunction | ∨ | OR | `P \| Q` | "P or Q" |
| Implication | ⇒ | IMPLIES | `P >> Q` | "if P then Q" |
| Biconditional | ⇔ | IFF | `P.iff(Q)` | "P if and only if Q" |

We use Python *operator overloading* so that logical sentences can be written much like mathematical notation.

> ⚠️ Mind Python's operator precedence: `>>` binds tighter than `&` and `|`, so `P & Q >> R` is parsed as `P & (Q >> R)`. Therefore **always use parentheses**, e.g. `(P & Q) >> R`.

In [ ]:
import itertools
from dataclasses import dataclass


class Expr:
    # Operator overloading so logical sentences are easy to write
    def __invert__(self):      return Not(self)            # ~P
    def __and__(self, other):  return And(self, other)     # P & Q
    def __or__(self, other):   return Or(self, other)      # P | Q
    def __rshift__(self, other): return Implies(self, other)  # P >> Q
    def iff(self, other):      return Iff(self, other)     # P.iff(Q)


@dataclass(frozen=True)
class Symbol(Expr):
    name: str
    def __repr__(self): return self.name


@dataclass(frozen=True)
class Not(Expr):
    arg: Expr
    def __repr__(self): return f"¬{self.arg}"


@dataclass(frozen=True)
class And(Expr):
    left: Expr
    right: Expr
    def __repr__(self): return f"({self.left} ∧ {self.right})"


@dataclass(frozen=True)
class Or(Expr):
    left: Expr
    right: Expr
    def __repr__(self): return f"({self.left} ∨ {self.right})"


@dataclass(frozen=True)
class Implies(Expr):
    left: Expr
    right: Expr
    def __repr__(self): return f"({self.left} ⇒ {self.right})"


@dataclass(frozen=True)
class Iff(Expr):
    left: Expr
    right: Expr
    def __repr__(self): return f"({self.left} ⇔ {self.right})"


def symbols(names):
    # symbols('P Q R') -> [P, Q, R]
    return [Symbol(n) for n in names.split()]

### 2.2 Building Sentences

`@dataclass(frozen=True)` makes every sentence **immutable** and **hashable**, so sentences can be stored in a `set` (useful for Resolution).

In [ ]:
P, Q, R = symbols('P Q R')

s1 = ~P
s2 = P & Q
s3 = (P | Q) >> R
s4 = (P >> Q).iff(~Q >> ~P)   # implication <=> contrapositive

for s in [s1, s2, s3, s4]:
    print(s)

## 3. Semantics: Models and Truth Tables

### 3.1 Models

A **model** assigns *True/False* to every proposition symbol, e.g. `{P: True, Q: False}`. The truth value of a complex sentence is computed recursively:

| P | Q | ¬P | P ∧ Q | P ∨ Q | P ⇒ Q | P ⇔ Q |
|---|---|---|---|---|---|---|
| F | F | T | F | F | **T** | T |
| F | T | T | F | T | **T** | F |
| T | F | F | F | T | F | F |
| T | T | F | T | T | T | T |

Note: `P ⇒ Q` is **false** only when P is true and Q is false. If the premise is false, the implication is considered true (*vacuously true*).

In [ ]:
def pl_true(s, model):
    # Evaluate sentence s in a model (dict Symbol -> bool)
    if isinstance(s, Symbol):
        return model[s]
    if isinstance(s, Not):
        return not pl_true(s.arg, model)
    if isinstance(s, And):
        return pl_true(s.left, model) and pl_true(s.right, model)
    if isinstance(s, Or):
        return pl_true(s.left, model) or pl_true(s.right, model)
    if isinstance(s, Implies):
        return (not pl_true(s.left, model)) or pl_true(s.right, model)
    if isinstance(s, Iff):
        return pl_true(s.left, model) == pl_true(s.right, model)
    raise TypeError(f"Not a logical sentence: {s!r}")


def prop_symbols(s):
    # Collect all proposition symbols in sentence s
    if isinstance(s, Symbol):
        return {s}
    if isinstance(s, Not):
        return prop_symbols(s.arg)
    return prop_symbols(s.left) | prop_symbols(s.right)


def all_models(syms):
    # Enumerate all 2^n models for the list of symbols syms
    for values in itertools.product([False, True], repeat=len(syms)):
        yield dict(zip(syms, values))

In [ ]:
print(pl_true(P >> Q, {P: True, Q: False}))   # False
print(pl_true(P >> Q, {P: False, Q: False}))  # True (vacuously true)
print(prop_symbols(s3))

### 3.2 Truth Tables

With `all_models` we can print the truth table of any sentence.

In [ ]:
def truth_table(*sentences):
    syms = sorted(set().union(*(prop_symbols(s) for s in sentences)), key=str)
    header = [str(x) for x in syms] + [str(s) for s in sentences]
    print(" | ".join(header))
    print("-" * len(" | ".join(header)))
    for model in all_models(syms):
        row = ["T" if model[x] else "F" for x in syms]
        row += ["T" if pl_true(s, model) else "F" for s in sentences]
        print(" | ".join(v.center(len(h)) for v, h in zip(row, header)))


truth_table(P >> Q, ~Q >> ~P, ~P | Q)

The columns `P ⇒ Q`, `¬Q ⇒ ¬P`, and `¬P ∨ Q` are always identical → the three sentences are **logically equivalent**. The equivalence `P ⇒ Q ≡ ¬P ∨ Q` is used later in CNF conversion.

### 3.3 Validity and Satisfiability

| Term | Definition | Example |
|---|---|---|
| **Valid** (tautology) | true in **all** models | `P ∨ ¬P` |
| **Satisfiable** | true in **at least one** model | `P ∧ Q` |
| **Unsatisfiable** (contradiction) | false in every model | `P ∧ ¬P` |

Key relationship: **α is valid ⇔ ¬α is unsatisfiable**. This is the basis of proof by contradiction (Resolution).

In [ ]:
def is_valid(s):
    syms = sorted(prop_symbols(s), key=str)
    return all(pl_true(s, m) for m in all_models(syms))


def is_satisfiable(s):
    syms = sorted(prop_symbols(s), key=str)
    return any(pl_true(s, m) for m in all_models(syms))


for s in [P | ~P, P & Q, P & ~P, ((P >> Q) & P) >> Q]:
    print(f"{str(s):30} valid={is_valid(s)!s:5}  satisfiable={is_satisfiable(s)}")

The last sentence `((P ⇒ Q) ∧ P) ⇒ Q` is valid. This is the **Modus Ponens** inference rule.

## 4. Entailment and TT-Entails

### 4.1 Entailment

**KB ⊨ α** (KB *entails* α) means: **in every model where KB is true, α is also true**.

In other words, `M(KB) ⊆ M(α)`.

### 4.2 The TT-Entails Algorithm (Model Checking)

TT-Entails (*Truth Table Entails*) recursively enumerates **all** models (AIMA Fig. 7.10):

1. Pick a symbol and branch on `True` and `False`.
2. Once every symbol has a value → check: if KB is true, then α **must** be true.
3. If KB is false in that model, the model is ignored (return `True`).

| Property | Value |
|---|---|
| Sound | Yes |
| Complete | Yes |
| Time complexity | O(2ⁿ), n = number of symbols |
| Space complexity | O(n) (DFS) |

In [ ]:
def tt_entails(kb, alpha):
    syms = sorted(prop_symbols(kb) | prop_symbols(alpha), key=str)
    return tt_check_all(kb, alpha, syms, {})


def tt_check_all(kb, alpha, syms, model):
    if not syms:                      # every symbol has a value
        if pl_true(kb, model):
            return pl_true(alpha, model)
        return True                   # KB is false -> model is irrelevant
    first, rest = syms[0], syms[1:]
    return (tt_check_all(kb, alpha, rest, {**model, first: True}) and
            tt_check_all(kb, alpha, rest, {**model, first: False}))

In [ ]:
kb = (P >> Q) & P
print("KB      :", kb)
print("KB |= Q :", tt_entails(kb, Q))    # True  (Modus Ponens)
print("KB |= R :", tt_entails(kb, R))    # False (R is not mentioned in the KB)

### 4.3 Inspecting the Models of a KB

To make this more concrete, the function below lists every model in which KB is true, together with the value of α in that model. If α is true in every row → KB ⊨ α.

In [ ]:
def show_models(kb, alpha):
    syms = sorted(prop_symbols(kb) | prop_symbols(alpha), key=str)
    print("Models in which KB is true:")
    for m in all_models(syms):
        if pl_true(kb, m):
            vals = ", ".join(f"{s}={'T' if m[s] else 'F'}" for s in syms)
            print(f"  {vals}   ->  {alpha} = {pl_true(alpha, m)}")


show_models((P | Q) & ~P, Q)

## 5. Conjunctive Normal Form (CNF)

### 5.1 Why CNF?

Resolution only works on sentences in **CNF**: a conjunction of clauses, where each **clause** is a disjunction of literals.

Example: `(A ∨ ¬B) ∧ (B ∨ C ∨ ¬D)`

### 5.2 Conversion Steps

| Step | Rule |
|---|---|
| 1. Eliminate ⇔ | `α ⇔ β` → `(α ⇒ β) ∧ (β ⇒ α)` |
| 2. Eliminate ⇒ | `α ⇒ β` → `¬α ∨ β` |
| 3. Move ¬ inwards | `¬¬α` → `α`; De Morgan: `¬(α ∧ β)` → `¬α ∨ ¬β`, `¬(α ∨ β)` → `¬α ∧ ¬β` |
| 4. Distribute ∨ over ∧ | `α ∨ (β ∧ γ)` → `(α ∨ β) ∧ (α ∨ γ)` |

In [ ]:
def eliminate_implications(s):
    # Steps 1 & 2
    if isinstance(s, Symbol):
        return s
    if isinstance(s, Not):
        return Not(eliminate_implications(s.arg))
    a, b = eliminate_implications(s.left), eliminate_implications(s.right)
    if isinstance(s, Iff):
        return And(Or(Not(a), b), Or(Not(b), a))
    if isinstance(s, Implies):
        return Or(Not(a), b)
    return type(s)(a, b)


def move_not_inwards(s):
    # Step 3
    if isinstance(s, Not):
        a = s.arg
        if isinstance(a, Symbol):
            return s
        if isinstance(a, Not):                        # ¬¬α = α
            return move_not_inwards(a.arg)
        if isinstance(a, And):                        # De Morgan
            return Or(move_not_inwards(Not(a.left)), move_not_inwards(Not(a.right)))
        if isinstance(a, Or):                         # De Morgan
            return And(move_not_inwards(Not(a.left)), move_not_inwards(Not(a.right)))
    if isinstance(s, (And, Or)):
        return type(s)(move_not_inwards(s.left), move_not_inwards(s.right))
    return s


def distribute_or_over_and(s):
    # Step 4
    if isinstance(s, And):
        return And(distribute_or_over_and(s.left), distribute_or_over_and(s.right))
    if isinstance(s, Or):
        a, b = distribute_or_over_and(s.left), distribute_or_over_and(s.right)
        if isinstance(a, And):        # (x ∧ y) ∨ b = (x ∨ b) ∧ (y ∨ b)
            return And(distribute_or_over_and(Or(a.left, b)), distribute_or_over_and(Or(a.right, b)))
        if isinstance(b, And):        # a ∨ (x ∧ y) = (a ∨ x) ∧ (a ∨ y)
            return And(distribute_or_over_and(Or(a, b.left)), distribute_or_over_and(Or(a, b.right)))
        return Or(a, b)
    return s


def to_cnf(s):
    return distribute_or_over_and(move_not_inwards(eliminate_implications(s)))

In [ ]:
B11, P12, P21 = symbols('B11 P12 P21')
s = B11.iff(P12 | P21)

print("Original   :", s)
s_a = eliminate_implications(s); print("Steps 1-2  :", s_a)
s_b = move_not_inwards(s_a);     print("Step 3     :", s_b)
s_c = distribute_or_over_and(s_b); print("Step 4     :", s_c)

### 5.3 Clause Representation

For Resolution, the CNF sentence is split into a **set of clauses**. Each clause is a `frozenset` of literals, e.g. `{¬B11, P12, P21}`. The empty clause `{}` (written □) means **contradiction**.

In [ ]:
def conjuncts(s):
    return conjuncts(s.left) + conjuncts(s.right) if isinstance(s, And) else [s]


def disjuncts(s):
    return disjuncts(s.left) + disjuncts(s.right) if isinstance(s, Or) else [s]


def negate(lit):
    return lit.arg if isinstance(lit, Not) else Not(lit)


def is_tautology(clause):
    # A clause containing both P and ¬P is always true, so it is useless for resolution
    return any(negate(l) in clause for l in clause)


def to_clauses(s):
    clauses = {frozenset(disjuncts(c)) for c in conjuncts(to_cnf(s))}
    return {c for c in clauses if not is_tautology(c)}


def clause_str(c):
    return "{" + ", ".join(sorted(map(str, c))) + "}" if c else "□"


for c in to_clauses(s):
    print(clause_str(c))

## 6. Resolution

### 6.1 The Resolution Rule

If one clause contains a literal `ℓ` and another contains `¬ℓ`, they can be combined by removing that complementary pair:

```
  (A ∨ B),   (¬B ∨ C)
  ───────────────────
        (A ∨ C)
```

### 6.2 Proof by Contradiction

To prove **KB ⊨ α**, we show that **KB ∧ ¬α is unsatisfiable**:

1. Convert `KB ∧ ¬α` into a set of clauses.
2. Resolve every pair of clauses and add the new resolvents.
3. If the **empty clause □** appears → contradiction → **KB ⊨ α** is proven.
4. If no new clauses can be added → **KB ⊭ α**.

Resolution is **sound** and **complete** (*refutation-complete*) for propositional logic.

In [ ]:
def pl_resolve(ci, cj):
    # All (non-tautological) resolvents of two clauses
    resolvents = []
    for lit in ci:
        if negate(lit) in cj:
            new = (ci - {lit}) | (cj - {negate(lit)})
            if not is_tautology(new):
                resolvents.append(frozenset(new))
    return resolvents


def pl_resolution(kb, alpha, verbose=False):
    clauses = to_clauses(kb & ~alpha)
    parents = {}                       # resolvent -> (parent clause 1, parent clause 2)
    while True:
        new = set()
        for ci, cj in itertools.combinations(list(clauses), 2):
            for r in pl_resolve(ci, cj):
                if r not in clauses and r not in new:
                    parents[r] = (ci, cj)
                    new.add(r)
                if not r:              # empty clause found
                    if verbose:
                        print_proof(r, parents)
                    return True
        if not new:                    # no new clauses
            return False
        clauses |= new


def print_proof(clause, parents, depth=0):
    # Print the proof tree from the empty clause back to the original clauses
    print("    " * depth + clause_str(clause))
    if clause in parents:
        for p in parents[clause]:
            print_proof(p, parents, depth + 1)

In [ ]:
kb = (P >> Q) & P
print("KB |= Q ?", pl_resolution(kb, Q, verbose=True))
print()
print("KB |= R ?", pl_resolution(kb, R))

How to read the proof tree: each indented line is a parent clause of the line above it. The empty clause □ comes from `{Q}` and `{¬Q}`, and `{¬Q}` comes from the negated query.

## 7. Forward Chaining (Horn Clauses)

### 7.1 Horn Clauses

A **definite clause** is a clause with **exactly one** positive literal, usually written as an implication:

```
A ∧ B ∧ C ⇒ D        (premises ⇒ conclusion)
A                    (fact)
```

For a KB containing only definite clauses, inference runs in **linear time** with **Forward Chaining**: start from the known facts, fire every rule whose premises are all satisfied, and repeat until the query is proven or no new facts appear.

### 7.2 Implementation (AIMA Fig. 7.15)

- `count[rule]` = number of premises not yet proven.
- `agenda` = queue of symbols known to be true.
- When a rule's `count` reaches 0 → its conclusion is added to the agenda.

In [ ]:
def pl_fc_entails(rules, facts, query, verbose=True):
    # rules : list of (premise_list, conclusion) ; facts : list of symbols
    count = {i: len(prem) for i, (prem, _) in enumerate(rules)}
    inferred = set()
    agenda = list(facts)
    while agenda:
        p = agenda.pop(0)
        if verbose:
            print(f"Processing: {p}")
        if p == query:
            return True
        if p in inferred:
            continue
        inferred.add(p)
        for i, (prem, concl) in enumerate(rules):
            if p in prem:
                count[i] -= 1
                if count[i] == 0:
                    if verbose:
                        print(f"   rule {' ∧ '.join(map(str, prem))} ⇒ {concl} fires")
                    agenda.append(concl)
    return False

Example from AIMA Fig. 7.16:

```
P ⇒ Q
L ∧ M ⇒ P
B ∧ L ⇒ M
A ∧ P ⇒ L
A ∧ B ⇒ L
A
B
```

In [ ]:
A, B, L, M = symbols('A B L M')

rules = [
    ([P], Q),
    ([L, M], P),
    ([B, L], M),
    ([A, P], L),
    ([A, B], L),
]
facts = [A, B]

print("\nKB |= Q ?", pl_fc_entails(rules, facts, Q))

## 8. Case Study: Wumpus World

### 8.1 Rules of the Wumpus World

In the Wumpus World (AIMA Section 7.2):
- Squares adjacent to a **Pit** are **Breezy**.
- Squares adjacent to the **Wumpus** are **Smelly** (Stench).
- The agent starts at [1,1] and can only perceive the square it is currently in.

Symbols used:
- `Pxy` : there is a pit at [x,y]
- `Bxy` : the agent perceives a breeze at [x,y]

### 8.2 Knowledge Base (AIMA Section 7.4.3)

The agent has visited [1,1] (no breeze) and then [2,1] (breeze).

| Rule | Sentence | Meaning |
|---|---|---|
| R1 | ¬P11 | There is no pit at [1,1] |
| R2 | B11 ⇔ (P12 ∨ P21) | [1,1] is breezy iff a neighbour has a pit |
| R3 | B21 ⇔ (P11 ∨ P22 ∨ P31) | [2,1] is breezy iff a neighbour has a pit |
| R4 | ¬B11 | Percept: no breeze at [1,1] |
| R5 | B21 | Percept: breeze at [2,1] |

In [ ]:
P11, P12, P21, P22, P31, B11, B21 = symbols('P11 P12 P21 P22 P31 B11 B21')

R1 = ~P11
R2 = B11.iff(P12 | P21)
R3 = B21.iff(P11 | P22 | P31)
R4 = ~B11
R5 = B21

wumpus_kb = R1 & R2 & R3 & R4 & R5
for i, r in enumerate([R1, R2, R3, R4, R5], 1):
    print(f"R{i}: {r}")

### 8.3 Inference with TT-Entails

The agent's questions:
1. Is [1,2] safe? (`¬P12`)
2. Does [2,2] contain a pit? (`P22`)
3. Is [2,2] safe? (`¬P22`)

In [ ]:
queries = [~P12, P12, P22, ~P22, P22 | P31]
for q in queries:
    print(f"KB |= {str(q):12} : {tt_entails(wumpus_kb, q)}")

**Analysis:**
- `KB ⊨ ¬P12` is **True**: from R4 (¬B11) and R2, `P12 ∨ P21` must be false, hence `¬P12` (and `¬P21`).
- `KB ⊨ P22` and `KB ⊨ ¬P22` are **both False**: the agent *does not know* whether [2,2] has a pit. There are models of the KB where P22 is true and others where it is false.
- `KB ⊨ P22 ∨ P31` is **True**: from R5 and R3 there must be a pit in one of the neighbours of [2,1], and P11 is already false by R1.

We can inspect the three models of the KB directly:

In [ ]:
show_models(wumpus_kb, P22)

### 8.4 Inference with Resolution

In [ ]:
for q in queries:
    print(f"KB |= {str(q):12} : {pl_resolution(wumpus_kb, q)}")

In [ ]:
# Proof tree for KB |= ¬P12
pl_resolution(wumpus_kb, ~P12, verbose=True)

Resolution gives the same results as TT-Entails, because both are **sound** and **complete**.

## 9. Comparison and Summary

| Aspect | TT-Entails | Resolution | Forward Chaining |
|---|---|---|---|
| Idea | Enumerate all models | Find a contradiction in KB ∧ ¬α | Derive new facts from rules |
| KB form | Any | CNF | Definite (Horn) clauses |
| Sound | Yes | Yes | Yes |
| Complete | Yes | Yes (refutation) | Yes (for Horn KBs) |
| Complexity | O(2ⁿ) | Exponential (worst case) | Linear |
| Strength | Simple, easy to understand | No model enumeration needed | Very fast |
| Weakness | Blows up for large n | Number of clauses can explode | Limited expressiveness |

## 10. Exercises (Homework)
<font color="blue">Exercise 6</font>

### Homework 1 — Validity, Satisfiability, and Equivalence

Complete the function `is_equivalent(a, b)`, which returns `True` if sentences `a` and `b` are logically equivalent.

**Hint:** `a ≡ b` if and only if the sentence `a ⇔ b` is **valid**.

In [ ]:
def is_equivalent(a, b):
    return _____


# Tests (expected: True, True, False)
print(is_equivalent(~(P & Q), ~P | ~Q))          # De Morgan
print(is_equivalent(P >> Q, ~Q >> ~P))           # Contrapositive
print(is_equivalent(P >> Q, Q >> P))             # Converse (not equivalent)

### Homework 2 — Knights and Knaves

On an island there are only two kinds of inhabitants:
- **Knights** always tell the truth.
- **Knaves** always lie.

You meet three inhabitants: A, B, and C.
- A says: *"B is a knave."*
- B says: *"A and C are the same kind."*

Use the symbols `A`, `B`, `C`, meaning "A is a knight", etc.

**Hint:** if X says statement S, then `X ⇔ S` (X is a knight if and only if S is true).

**Tasks:**
1. Complete `R_A` and `R_B`.
2. Use `tt_entails` **and** `pl_resolution` to determine what C is.
3. Can we determine what A is? Explain using `show_models`.

In [ ]:
A, B, C = symbols('A B C')

R_A = _____   # A's statement
R_B = _____   # B's statement

kk_kb = R_A & R_B

for q in [A, ~A, B, ~B, C, ~C]:
    print(f"KB |= {str(q):3} | TT: {tt_entails(kk_kb, q)!s:5} | Resolution: {pl_resolution(kk_kb, q)}")

In [ ]:
show_models(kk_kb, ~C)

**Analysis (write your answer here):**

- C is a: ...
- Can we determine what A is? ...
- Reasoning (use mathematical logic): ...